# 03 · Guardrails — 02 Tool Guard

**Everything here runs offline with no API key.** No model is called at any point in
this notebook; the guards are pure regex and set membership, and the "tool" is a
local function with a side-effect counter.

**In → out:** a user query plus a proposed tool call goes in. Out comes either the
tool's real result, or a refusal JSON blob the tool body never saw.

---

## The boundary against `01-tools/05-gate`

This stage and `01-modules/01-tools/05-gate` are both called "checks", and
contributors reliably build one when they meant the other. They are not the same
thing and they do not run at the same time:

> **`01-tools/05-gate` judges EVIDENCE THAT CAME BACK.** It inspects a produced
> answer against retrieved context and scores whether it is grounded. The answer
> already exists by the time it runs; the model has already been paid; the
> retrieval has already happened. Its output is a verdict *about* work that is
> finished.
>
> **`03-guardrails` — this stage — STOPS AN ACTION BEFORE IT HAPPENS.** It blocks an
> input from being processed, or a tool call from running at all. When it says no,
> there is no answer to judge, because nothing ran.

Put the two on a timeline and the distinction is unmissable:

| | `01-tools/05-gate` | `03-guardrails` (here) |
|---|---|---|
| Runs | **after** the work | **before** the work |
| Input | an answer + the context it should have come from | a query + a proposed action |
| Output | a grounding score, flagged quotes, orphan citations | permit / refuse |
| Cost when it fires | the answer was already generated and paid for | zero — the thing never ran |
| Worst case if it is wrong | a bad answer is shown, or a good one withheld | an unsafe action executes, or a safe one is blocked |
| Reversible? | yes — withhold the answer | **no** — an executed tool call has already touched the world |

The last row is the whole reason this stage exists separately. A grounding check
can afford to be late, because withholding an answer after generating it still
works — you paid for it, but nothing irreversible happened. A tool guard cannot
afford to be late, because a tool call that has already run has already sent the
email, written the row, or produced the clinical number a reader will act on.
"Undo" is not a feature a guardrail gets to rely on.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `set_user_query` / `get_user_query` | Request-scoped storage of the **original** user text, via `contextvars`, so a guard deep in a call stack can see what was actually asked. | `set_user_query("...")` → `get_user_query()` |
| `classify_query_intents` | Regex intent classification of a query. Returns a frozenset — a query can have several intents. | `"stage T2 N1 breast cancer"` → `{TNM_STAGING}` |
| `allowed_calculator_tool_names` | Which tools the model is even permitted to see for this query. | comparative-evidence query → `frozenset()` |
| `_reject` | Builds the refusal JSON, with a reason and a directive telling the model what to do instead. | → `{"blocked": true, "tool": ..., "reason": ...}` |
| `guard_tram_flap_selection` | Returns refusal JSON if the call must not run; `None` if it may. Also blocks invented parameters. | BMI not in the query → refused |
| `guarded` | Wraps a tool so the guard runs before the body. | `guarded(guard, tool)` |
| `TOOL_CALLS` | A side-effect counter inside the tool body. Proves the body ran, or did not. | stays at `0` on a refusal |

## Ported from

`clinical-search`'s `services/tool_guardrails.py`, with its two dependencies —
`services/tool_intent.py` and `services/query_context.py` — carried across with it,
since the guard is meaningless without them. Regexes, intent rules and rejection
messages are the donor's. All patient scenarios below are invented for this notebook.

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio
nbio.bootstrap()

In [ ]:
nbio.show_environment()
print("\nNo model is called in this notebook. Nothing below depends on a key.")

## Step 1 — the original user query, available anywhere in the stack

Every guard in this file asks the same question: *did the user actually say this?*
That is only answerable if the original, unmodified user text is reachable from
wherever the guard runs — which is typically deep inside a tool-calling loop, several
frames below whoever received the request.

The donor uses a `contextvars.ContextVar`, not a global and not a parameter threaded
through every signature. The reason is concurrency: under a server handling
overlapping requests, a module-level global would let one request's query leak into
another request's guard decision. A `ContextVar` is scoped per execution context.

In [ ]:
import contextvars

_query_var: contextvars.ContextVar[str | None] = contextvars.ContextVar(
    "guardrails_user_query", default=None
)


def set_user_query(query: str | None) -> None:
    _query_var.set((query or "").strip() or None)


def get_user_query() -> str:
    return (_query_var.get() or "").strip()


set_user_query("  Is DIEP oncologically safe compared to TRAM?  ")
print(f"stored query: {get_user_query()!r}")

set_user_query(None)
print(f"after clearing: {get_user_query()!r}")

assert get_user_query() == "", "a cleared context must read as empty, never as None"

## Step 2 — intent classification, entirely by regex

Five intents. A query may carry more than one, so the result is a `frozenset`, not a
single label. Read the last rule in `classify_query_intents` carefully — it is the
one doing the security work:

> A comparative-evidence question must **not** also expose patient-selection
> calculators unless the query explicitly asks for both.

"Is DIEP oncologically safe compared to TRAM?" is a literature question about two
techniques in general. It is not a request to pick a flap for a specific person. If
the patient-selection calculator were visible for it, the model would have to invent
a patient — a BMI, a smoking status — to fill in the arguments.

In [ ]:
import re
from enum import Enum
from typing import FrozenSet


class QueryIntent(str, Enum):
    COMPARATIVE_EVIDENCE = "comparative_evidence"
    GENERAL_EVIDENCE = "general_evidence"
    PATIENT_SELECTION = "patient_selection"
    NAC_PLANNING = "nac_planning"
    TNM_STAGING = "tnm_staging"


_COMPARATIVE_RE = re.compile(
    r"""
    \b(vs\.?|versus|compared\s+to|comparison\s+of)\b
    | \b(complication(s)?|morbidity|outcomes?|efficacy|safety|meta-?analysis|systematic\s+review)\b
    .*\b(vs\.?|versus|compared|between|difference)\b
    | \b(oncologically\s+safe|is\s+.+\s+safe)\b
    """,
    re.I | re.X,
)

_PATIENT_SELECTION_RE = re.compile(
    r"""
    \b(should\s+(we|i|the\s+team|this\s+patient)|which\s+flap\s+(should|is\s+best)|candidate\s+for)
    | \b(for\s+a\s+patient|this\s+patient|year-?old\s+patient)
    | \b(select(ing|ion)\s+(a|the)\s+flap|choose\s+between\s+DIEP\s+and\s+TRAM)
    """,
    re.I | re.X,
)

_NAC_RE = re.compile(
    r"""
    \b(nac\s+position|nipple.?areolar\s+complex|areola\s+diameter|notch.?to.?nipple
    | nipple\s+position|symmetry\s+mark(ing)?|ptosis\s+grade|regnault)
    """,
    re.I | re.X,
)

_TNM_RE = re.compile(
    r"""
    \b(tnm|t\d\s+n\d|ajcc|pathologic\s+stage|clinical\s+stage|stage\s+[i1234]|staging\s+for)
    """,
    re.I | re.X,
)

# Parameter-provenance patterns: each asks "is this value present in what the user
# actually typed?" They are what stops a model filling an argument from thin air.
_BMI_IN_QUERY_RE = re.compile(r"\b(bmi|body\s+mass\s+index|\d+\s*kg/m)\b", re.I)
_SMOKER_IN_QUERY_RE = re.compile(r"\b(smok(er|ing|es)|tobacco|cigarette)\b", re.I)
_PRIOR_ABD_SURGERY_RE = re.compile(
    r"\b(prior\s+(abdominal|laparotomy|c-?section|cesarean)|previous\s+abdominal|laparotomy)\b",
    re.I,
)
_LARGE_VOLUME_RE = re.compile(r"\b(large\s+volume|substantial\s+volume|high\s+volume\s+need)\b", re.I)


def classify_query_intents(query: str) -> frozenset[QueryIntent]:
    """Return all intents detected for a user query (may be multiple)."""
    q = (query or "").strip()
    if not q:
        return frozenset({QueryIntent.GENERAL_EVIDENCE})

    intents: set[QueryIntent] = set()

    if _COMPARATIVE_RE.search(q) and not _PATIENT_SELECTION_RE.search(q):
        intents.add(QueryIntent.COMPARATIVE_EVIDENCE)

    if _PATIENT_SELECTION_RE.search(q) or (
        _BMI_IN_QUERY_RE.search(q) and re.search(r"\b(flap|tram|diep|reconstruction)\b", q, re.I)
    ):
        intents.add(QueryIntent.PATIENT_SELECTION)

    if _NAC_RE.search(q):
        intents.add(QueryIntent.NAC_PLANNING)

    if _TNM_RE.search(q):
        intents.add(QueryIntent.TNM_STAGING)

    if not intents:
        intents.add(QueryIntent.GENERAL_EVIDENCE)

    # Comparative questions must not also expose patient-selection calculators unless
    # the query explicitly asks for both (rare).
    if QueryIntent.COMPARATIVE_EVIDENCE in intents and QueryIntent.PATIENT_SELECTION not in intents:
        intents.discard(QueryIntent.PATIENT_SELECTION)

    return frozenset(intents)

## Step 3 — what each synthetic query classifies as

Four invented queries, chosen so that each lands on a different branch. Everything
here is made up for the notebook — no real patient, no real case.

In [ ]:
QUERIES = {
    "comparative": "Is DIEP oncologically safe compared to TRAM?",
    "selection": "For a patient with BMI 34 who is a smoker with prior abdominal surgery, which flap should we choose?",
    "nac": "What notch-to-nipple distance should I mark for symmetry at 21 cm breast base width?",
    "staging": "What is the AJCC pathologic stage for a T2 N1 M0 breast cancer?",
}

rows = []
for label, q in QUERIES.items():
    intents = sorted(i.value for i in classify_query_intents(q))
    rows.append((label, q[:52] + ("..." if len(q) > 52 else ""), ", ".join(intents)))

nbio.table(rows, headers=("label", "query", "intents"))

assert classify_query_intents(QUERIES["comparative"]) == frozenset({QueryIntent.COMPARATIVE_EVIDENCE})
assert QueryIntent.PATIENT_SELECTION in classify_query_intents(QUERIES["selection"])
assert QueryIntent.PATIENT_SELECTION not in classify_query_intents(QUERIES["comparative"]), \
    "a general comparison must never look like a request to pick a flap for someone"

## Step 4 — which tools the model is even allowed to see

`allowed_calculator_tool_names` is the first of the two layers. Note its early
return: for a purely comparative or general-evidence query it returns an **empty**
frozenset, meaning the model is offered no calculators at all. The safest tool call
is the one the model was never shown a button for.

`filter_domain_tools` applies that to a real tool list. This is the cheapest guard in
the file and the one that prevents the most trouble, because it removes the
temptation rather than punishing it.

In [ ]:
def allowed_calculator_tool_names(query: str) -> FrozenSet[str]:
    """Tool function names the model may see for this query."""
    intents = classify_query_intents(query)
    allowed: set[str] = set()

    if QueryIntent.COMPARATIVE_EVIDENCE in intents or QueryIntent.GENERAL_EVIDENCE in intents:
        if QueryIntent.PATIENT_SELECTION not in intents:
            return frozenset()

    if QueryIntent.PATIENT_SELECTION in intents:
        allowed.add("tram_flap_selection_tool")

    if QueryIntent.NAC_PLANNING in intents:
        allowed.add("nac_positioning_tool")

    if QueryIntent.TNM_STAGING in intents:
        allowed.add("tnm_staging_tool")

    return frozenset(allowed)


def tool_name(obj: object) -> str:
    name = getattr(obj, "__name__", None) or getattr(obj, "name", None)
    return str(name or "")


def filter_domain_tools(tools: list, query: str) -> list:
    """Return only the domain calculators permitted for this query."""
    allowed = allowed_calculator_tool_names(query)
    if not allowed:
        return []
    return [t for t in tools if tool_name(t) in allowed]


nbio.table(
    [(label, ", ".join(sorted(allowed_calculator_tool_names(q))) or "(none offered)")
     for label, q in QUERIES.items()],
    headers=("label", "tools the model is shown"),
)

assert allowed_calculator_tool_names(QUERIES["comparative"]) == frozenset()
assert allowed_calculator_tool_names(QUERIES["selection"]) == {"tram_flap_selection_tool"}
assert allowed_calculator_tool_names(QUERIES["staging"]) == {"tnm_staging_tool"}

## Step 5 — the refusal, and why it is not an exception

`_reject` returns a JSON **string**, not a raised error, and this is a deliberate
difference from `01-input-guard.ipynb`, where the scope gate raises.

The reason is who the reader is. A scope gate's reader is your own calling code,
which should stop. A tool guard's reader is *the model*, mid-conversation: the
refusal goes back into the transcript as that tool call's result. So it carries a
`directive` — a plain instruction about what to do instead. A bare exception, or an
empty result, tends to produce a model that retries the same blocked call with
slightly different arguments.

In [ ]:
import json

_BLOCKED_RESPONSE = (
    "Calculator skipped: this question is answered from retrieved literature only. "
    "Do not invent patient parameters. Synthesize directly from the provided papers."
)

# Refusals are counted here purely so the notebook can assert on them; the donor
# logs instead.
REJECTIONS: list[dict] = []


def _reject(reason: str, *, tool: str) -> str:
    """Return the blocked-call payload the model sees in place of a tool result."""
    REJECTIONS.append({"tool": tool, "reason": reason})
    return json.dumps(
        {
            "blocked": True,
            "tool": tool,
            "reason": reason,
            "directive": _BLOCKED_RESPONSE,
        }
    )


print(json.dumps(json.loads(_reject("example reason", tool="demo_tool")), indent=2))
REJECTIONS.clear()

## Step 6 — the three guards

Each returns refusal JSON if the call must not run, and `None` if it may. `None`
meaning "permitted" reads backwards at first glance; it is the donor's convention and
it is kept, because it makes the call site a single clean line:
`if (blocked := guard(...)): return blocked`.

Two distinct checks live in `guard_tram_flap_selection`, and they fail for different
reasons:

1. **Is this tool permitted for this query's intent at all?** (the Step 4 layer again,
   re-checked at call time — a model that was never offered the tool can still emit a
   call for it)
2. **Did each argument's value actually appear in the user's query?** BMI, smoking
   status, prior abdominal surgery and volume need are each checked against the
   original text. A model that is confident and wrong about a patient's BMI produces a
   clinical number that looks entirely legitimate downstream.

In [ ]:
def guard_tram_flap_selection(
    *,
    prior_abdominal_surgery: bool,
    smoker: bool,
    bmi: float,
    need_large_volume: bool,
) -> str | None:
    """Return error JSON if call should not run; None if OK."""
    tool = "tram_flap_selection_tool"
    query = get_user_query()
    allowed = allowed_calculator_tool_names(query)

    if tool not in allowed:
        intents = ", ".join(i.value for i in classify_query_intents(query))
        return _reject(
            f"Tool not permitted for query intent ({intents}). "
            "Use for explicit patient flap-selection scenarios only.",
            tool=tool,
        )

    q = query.lower()
    if not _BMI_IN_QUERY_RE.search(q) and bmi > 0:
        return _reject("BMI was not provided in the user query; do not invent patient BMI.", tool=tool)
    if smoker and not _SMOKER_IN_QUERY_RE.search(q):
        return _reject("Smoking status was not in the user query.", tool=tool)
    if prior_abdominal_surgery and not _PRIOR_ABD_SURGERY_RE.search(q):
        return _reject("Prior abdominal surgery was not mentioned in the user query.", tool=tool)
    if need_large_volume and not _LARGE_VOLUME_RE.search(q):
        return _reject("Large volume need was not stated in the user query.", tool=tool)

    return None


def guard_nac_positioning(*, breast_base_width_cm: float, notch_to_nipple_cm: float) -> str | None:
    tool = "nac_positioning_tool"
    query = get_user_query()
    if tool not in allowed_calculator_tool_names(query):
        return _reject(
            "NAC positioning tool only for queries about nipple/areola positioning or symmetry marking.",
            tool=tool,
        )
    q = query.lower()
    has_measure = re.search(r"\b(\d+\.?\d*)\s*(cm|centimeter)\b", q)
    has_nac_topic = re.search(r"\b(nac|nipple|areola|notch)\b", q, re.I)
    if not has_nac_topic:
        return _reject("Query does not ask about NAC/nipple positioning.", tool=tool)
    if not has_measure and (breast_base_width_cm != 0 or notch_to_nipple_cm != 0):
        return _reject("Measurements must come from the user query, not invented values.", tool=tool)
    return None


def guard_tnm_staging(*, t: str, n: str, m: str) -> str | None:
    tool = "tnm_staging_tool"
    query = get_user_query()
    if tool not in allowed_calculator_tool_names(query):
        return _reject("TNM tool only when query asks for cancer staging.", tool=tool)
    if not re.search(r"\b(t[0-4x]?|n[0-3x]?|m[01x]?|stage|tnm)\b", query, re.I):
        return _reject("Staging values not grounded in user query.", tool=tool)
    return None

## Step 7 — a tool body that leaves evidence

The claim this notebook has to prove is *"the refused call never reached the tool
body."* Asserting that the return value looks like a refusal does not prove it — a
tool could run, touch whatever it touches, and then have its result discarded by a
wrapper, and the return value would look identical.

So the tool body increments a module-level counter and appends to a log as its
**first statement**. The counter is the evidence. If it is still `0` after a refused
call, the body provably did not execute — there is no path through it that leaves the
counter untouched.

In [ ]:
TOOL_CALLS = 0
TOOL_LOG: list[dict] = []


def tram_flap_selection_tool(
    *,
    prior_abdominal_surgery: bool,
    smoker: bool,
    bmi: float,
    need_large_volume: bool,
) -> str:
    """Stand-in for the real calculator. Records that it ran before doing anything
    else, so a refused call is provably distinguishable from a discarded result."""
    global TOOL_CALLS
    TOOL_CALLS += 1
    TOOL_LOG.append(
        {"bmi": bmi, "smoker": smoker, "prior_abd": prior_abdominal_surgery,
         "large_volume": need_large_volume}
    )

    risk = 0
    if bmi >= 30:
        risk += 1
    if smoker:
        risk += 1
    if prior_abdominal_surgery:
        risk += 1
    verdict = "free TRAM / DIEP with caution" if risk >= 2 else "pedicled TRAM acceptable"
    return json.dumps({"blocked": False, "risk_factors": risk, "recommendation": verdict})


def guarded(guard, tool):
    """Run the guard first; call the tool body only if the guard returns None."""
    def wrapper(**kwargs):
        blocked = guard(**kwargs)
        if blocked is not None:
            return blocked
        return tool(**kwargs)

    wrapper.__name__ = tool_name(tool)
    return wrapper


guarded_tram = guarded(guard_tram_flap_selection, tram_flap_selection_tool)
print(f"wrapper preserves the tool name for filtering: {tool_name(guarded_tram)!r}")
assert tool_name(guarded_tram) == "tram_flap_selection_tool"

## Step 8 — the permitted call: it goes all the way through

The patient-selection query names a BMI, a smoking status and prior abdominal
surgery. Every argument is traceable to the user's own words, so the guard returns
`None` and the body runs.

Counter before and after are printed, so "it ran" is a measurement, not a claim.

In [ ]:
TOOL_CALLS = 0
TOOL_LOG.clear()
REJECTIONS.clear()

set_user_query(QUERIES["selection"])
print(f"query: {get_user_query()}\n")

before = TOOL_CALLS
result = guarded_tram(
    prior_abdominal_surgery=True,
    smoker=True,
    bmi=34.0,
    need_large_volume=False,
)
after = TOOL_CALLS

print(f"TOOL_CALLS before: {before}   after: {after}")
print("result:")
nbio.show_json(json.loads(result))

assert after == before + 1, "a permitted call must actually reach the tool body"
assert json.loads(result)["blocked"] is False
assert TOOL_LOG == [{"bmi": 34.0, "smoker": True, "prior_abd": True, "large_volume": False}]
assert REJECTIONS == []

## Step 9 — the refused call: the body never runs

Same tool, same arguments. The only thing that changes is the **query in context** —
now the general comparative question, which is answered from literature and has no
patient in it at all.

The counter is reset to `0` first, and the assertion is on the counter, not on the
return value.

In [ ]:
TOOL_CALLS = 0
TOOL_LOG.clear()
REJECTIONS.clear()

set_user_query(QUERIES["comparative"])
print(f"query: {get_user_query()}\n")

result = guarded_tram(
    prior_abdominal_surgery=True,
    smoker=True,
    bmi=34.0,
    need_large_volume=False,
)

print(f"TOOL_CALLS after a refused call: {TOOL_CALLS}")
print("what the model gets back instead of a result:")
nbio.show_json(json.loads(result))

# The counter is the proof. The tool body increments it as its first statement,
# so there is no way to execute any part of the body and leave it at zero.
assert TOOL_CALLS == 0, "the tool body ran despite the guard -- this is the failure this notebook exists to catch"
assert TOOL_LOG == [], "the tool body recorded nothing, because it never started"
assert json.loads(result)["blocked"] is True
assert len(REJECTIONS) == 1
print(f"\nrejection reason: {REJECTIONS[0]['reason']}")

## Step 10 — the invented-parameter case

This one is subtler than the intent block, and it is the check most likely to be
dropped by someone porting this code in a hurry.

The query below *is* a legitimate patient-selection question — the tool is permitted.
But the user never mentioned a BMI. A model filling in `bmi=34.0` anyway has
fabricated a clinical measurement, and the calculator would dutifully turn that
fabrication into a risk score and a recommendation that reads exactly like a real one.

The guard blocks the call on argument provenance alone, after having already decided
the tool itself was allowed.

In [ ]:
TOOL_CALLS = 0
REJECTIONS.clear()

set_user_query("Which flap should we choose for this patient before mastectomy?")
print(f"query: {get_user_query()}")
print(f"tool permitted for this query: {allowed_calculator_tool_names(get_user_query())}\n")

result = guarded_tram(
    prior_abdominal_surgery=False,
    smoker=False,
    bmi=34.0,          # never appears in the query above
    need_large_volume=False,
)

print(f"TOOL_CALLS: {TOOL_CALLS}")
print(f"reason    : {json.loads(result)['reason']}")

assert allowed_calculator_tool_names(get_user_query()) == {"tram_flap_selection_tool"}, \
    "the tool IS permitted here -- the block below is about the argument, not the tool"
assert TOOL_CALLS == 0
assert "do not invent patient bmi" in json.loads(result)["reason"].lower()

## Step 11 — every guard, every query, in one grid

Twelve combinations: three guards against four queries. `PERMIT` should appear
exactly three times — on the diagonal where the query's intent matches the tool.

Note the NAC and TNM columns for the `selection` query: those tools are blocked there
even though that query is a genuine patient-selection question, because being
permitted for *one* tool is not being permitted for *all* of them.

In [ ]:
TOOL_CALLS = 0
REJECTIONS.clear()

CASES = [
    ("tram_flap_selection_tool", lambda: guard_tram_flap_selection(
        prior_abdominal_surgery=True, smoker=True, bmi=34.0, need_large_volume=False)),
    ("nac_positioning_tool", lambda: guard_nac_positioning(
        breast_base_width_cm=21.0, notch_to_nipple_cm=19.0)),
    ("tnm_staging_tool", lambda: guard_tnm_staging(t="T2", n="N1", m="M0")),
]

rows = []
permits = 0
for label, q in QUERIES.items():
    set_user_query(q)
    verdicts = []
    for tool, call in CASES:
        blocked = call()
        if blocked is None:
            verdicts.append("PERMIT")
            permits += 1
        else:
            verdicts.append("refuse")
    rows.append((label, *verdicts))

nbio.table(rows, headers=("query", "tram_flap", "nac_positioning", "tnm_staging"))

print(f"\npermits: {permits} of {len(QUERIES) * len(CASES)} combinations")
print(f"refusals recorded: {len(REJECTIONS)}")
print(f"TOOL_CALLS across the whole grid: {TOOL_CALLS}  (no tool body was invoked -- guards only)")

assert permits == 3, permits
assert len(REJECTIONS) == 9
assert TOOL_CALLS == 0

## Step 12 — the cheapest layer: never offer the tool at all

Steps 8–11 exercise the second layer, which catches a call the model has already
decided to make. `filter_domain_tools` is the first layer: for a comparative query,
the calculators are simply absent from the tool list handed to the model.

Both layers are needed, and the redundancy is intentional. Layer one means the model
is rarely tempted. Layer two means that when a model emits a call for a tool it was
never shown — which happens, from a stale transcript or a cached schema — the call
still does not execute.

In [ ]:
def nac_positioning_tool(**kwargs) -> str:
    return json.dumps({"blocked": False, "note": "stand-in"})


def tnm_staging_tool(**kwargs) -> str:
    return json.dumps({"blocked": False, "note": "stand-in"})


ALL_TOOLS = [guarded_tram, nac_positioning_tool, tnm_staging_tool]

for label in QUERIES:
    set_user_query(QUERIES[label])
    offered = filter_domain_tools(ALL_TOOLS, get_user_query())
    print(f"{label:<12} model is offered: {sorted(tool_name(t) for t in offered) or '(no calculators)'}")

set_user_query(QUERIES["comparative"])
assert filter_domain_tools(ALL_TOOLS, get_user_query()) == []
set_user_query(QUERIES["staging"])
assert [tool_name(t) for t in filter_domain_tools(ALL_TOOLS, get_user_query())] == ["tnm_staging_tool"]

print("\nLayer 1 removes the temptation. Layer 2 (Step 9) stops the call anyway.")

## Wrap-up

A permitted call reached the tool body and `TOOL_CALLS` went to `1`. A refused call,
with byte-identical arguments, left `TOOL_CALLS` at `0` and `TOOL_LOG` empty — the
body did not run, and that is measured rather than asserted in prose. The only thing
that differed between the two was the user query sitting in the context variable.

Three things worth carrying to any tool guard you write:

- **Guard on argument provenance, not just tool identity.** Step 10's call was to a
  permitted tool, by a model doing something reasonable-looking, and still had to be
  stopped. "Which tool" and "with what values" are separate questions.
- **Refuse with a directive, not an exception.** The refusal is read by a model that
  is still mid-turn and will do *something* next. Tell it what.
- **Prove the body did not run.** A side-effect counter costs three lines and is the
  difference between a guard you have tested and a guard you have described.

And the boundary this notebook opened with, restated now that both sides have been
seen: `05-gate` scores an answer that already exists. This stage stops the action
before it exists. Reach for `05-gate` when a wrong output is the risk; reach for this
stage when a wrong *action* is.

Next: `03-fail-closed.ipynb` — what these guards should do when the guard itself
breaks, and a real defect from our own lab that gets it backwards.

## What did not come across

- **The Strands `@tool` decorator.** The donor's tools are registered with an agent
  framework that turns a typed signature into a JSON schema the model sees. Plain
  functions here — this stage shows what a guard does, not what a framework does.
- **`logger.info("tool guardrail blocked ...")`.** The donor logs every rejection to
  a real logging pipeline. The `REJECTIONS` list is a notebook-local stand-in so the
  cells can assert on what was blocked.
- **The actual clinical calculators.** `tram_flap_selection_tool` here is a
  three-line risk count invented for the notebook. The real flap-selection,
  NAC-positioning and TNM-staging logic is clinical content, not guardrail machinery,
  and does not belong in a teaching repo.
- **Where the context variable gets set.** In the donor, `set_user_query` is called
  once by the request handler at the top of a turn. That handler is web-framework
  code; here every step sets it explicitly so you can see which query each guard is
  reading.
- **Any model in the loop.** No step here asks a model to decide anything. That is
  the point — a guardrail that needs a model call to decide whether to permit a model
  call has a bootstrapping problem, and a bill.